## **Library**

In [16]:
import pandas as pd
import re
import numpy as np
import ast
import pymatgen
from pymatgen.core import Structure, Lattice
from pymatgen.symmetry.analyzer import SpacegroupAnalyzer
from pymatgen.core.structure_matcher import StructureMatcher


# **Data Handling**

In [17]:

###########################################################################################################
################################################ STEP 1 ###################################################
###########################################################################################################


df0 = pd.read_csv(
    "llm_crystal_benchmark_dataset.csv",
    engine="python",
    sep=",",
    quotechar='"'
)


###########################################################################################################
############################################### STEP 2 ####################################################
###########################################################################################################

df1 = df0

def clean_space_group(sg):
    if isinstance(sg, str):
        return re.sub(r"\s*\(.*?\)", "", sg).strip()
    return sg

df1["llm_space_group"] = df1["llm_space_group"].apply(clean_space_group)
df1["mp_space_group"]  = df1["mp_space_group"].apply(clean_space_group)


###########################################################################################################
############################################### STEP 3 ####################################################
###########################################################################################################


df=df1

df["llm_lattice_params"] = df["llm_lattice_params"].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
)

df["mp_lattice_params"] = df["mp_lattice_params"].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
)

df["llm_atomic_positions"] = df["llm_atomic_positions"].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
)

df["mp_atomic_positions"] = df["mp_atomic_positions"].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
)


# **Similarity Scoring**

In [24]:
import numpy as np
from pymatgen.core import Structure, Lattice
from pymatgen.analysis.structure_matcher import StructureMatcher
from pymatgen.symmetry.analyzer import SpacegroupAnalyzer

def Score(row, k=1.0):
    print()
    print(f"DEBUG Row {row.name}:")
    actual_score = 0.0

    if not row.get("success", False):
        print("Insuss")
        return 0.0
    actual_score = 0.25
    print("Sucess")

    if str(row.get("llm_formula")) != str(row.get("mp_formula")):
        print("Different formula")
        return actual_score
    actual_score += 0.25
    print("Same formula")

    llm_sg = row.get("llm_space_group")
    mp_sg  = row.get("mp_space_group")

    if llm_sg != mp_sg:
        print("Different space group")
        return actual_score
    actual_score += 0.25
    print("Same Space group")

    try:
        # =========================
        # --- MP ---
        # =========================
        mp_lat = list(row["mp_lattice_params"].values())
        mp_el  = [d["element"] for d in row["mp_atomic_positions"]]
        mp_pos = [d["position"] for d in row["mp_atomic_positions"]]

        mp_struct = Structure(Lattice.from_parameters(*mp_lat), mp_el, mp_pos)

        mp_sg2 = SpacegroupAnalyzer(mp_struct, symprec=1e-2).get_space_group_symbol()
        print("Raw MP SG match:", mp_sg == mp_sg2)

        if mp_sg2 != mp_sg:
            try:
                mp_struct = Structure.from_spacegroup(
                    mp_sg,
                    Lattice.from_parameters(*mp_lat),
                    mp_el,
                    mp_pos
                )
                mp_struct.merge_sites(tol=0.1, mode="delete")
            except:
                pass

        # =========================
        # --- LLM ---
        # =========================
        llm_lat = list(row["llm_lattice_params"].values())
        llm_el  = [d["element"] for d in row["llm_atomic_positions"]]
        llm_pos = [d["position"] for d in row["llm_atomic_positions"]]

        llm_struct = Structure(Lattice.from_parameters(*llm_lat), llm_el, llm_pos)

        llm_sg2 = SpacegroupAnalyzer(llm_struct, symprec=1e-2).get_space_group_symbol()
        print("RAW LLM SG match:", llm_sg == llm_sg2)

        if llm_sg2 != llm_sg:
            try:
                llm_struct = Structure.from_spacegroup(
                    llm_sg,
                    Lattice.from_parameters(*llm_lat),
                    llm_el,
                    llm_pos
                )
                llm_struct.merge_sites(tol=0.1, mode="delete")
            except:
                pass

        # =========================
        # --- PRIMITIVE CHECK ---
        # =========================
        try:
            mp_prim  = mp_struct.get_primitive_structure()
            llm_prim = llm_struct.get_primitive_structure()


            # =========================
            # --- For Debbuging  ---
            # =========================

            n_mp_prim  = len(mp_prim)
            n_llm_prim = len(llm_prim)

            if n_mp_prim != n_llm_prim:
                print(f"DEBUG Row {row.name}: Primitive mismatch (MP: {n_mp_prim} vs LLM: {n_llm_prim})")

        except Exception as e:
            print(f"DEBUG Row {row.name}: Primitive error: {e}")

        # =========================
        # --- MATCHING ---
        # =========================
        matcher = StructureMatcher(stol=1)

        res = matcher.get_rms_dist(mp_struct, llm_struct)

        if res:
            rms_val = res[0] if isinstance(res, tuple) else res
            actual_score += 0.25 * np.exp(-k * rms_val)

    except Exception as e:
        print(f"DEBUG Row {row.name}: Error: {e}")

    return round(actual_score, 4)


df['score'] = df.apply(lambda r: Score(r, k=1), axis=1)
df


DEBUG Row 0:
Sucess
Same formula
Same Space group
Raw MP SG match: True
RAW LLM SG match: False

DEBUG Row 1:
Sucess
Same formula
Different space group

DEBUG Row 2:
Sucess
Same formula
Same Space group
Raw MP SG match: True
RAW LLM SG match: True

DEBUG Row 3:
Sucess
Same formula
Same Space group
Raw MP SG match: True
RAW LLM SG match: False

DEBUG Row 4:
Sucess
Same formula
Same Space group
Raw MP SG match: True
RAW LLM SG match: True

DEBUG Row 5:
Sucess
Same formula
Same Space group
Raw MP SG match: True
RAW LLM SG match: False

DEBUG Row 6:
Sucess
Same formula
Same Space group
Raw MP SG match: True
RAW LLM SG match: False

DEBUG Row 7:
Sucess
Same formula
Same Space group
Raw MP SG match: True
RAW LLM SG match: True

DEBUG Row 8:
Sucess
Same formula
Different space group

DEBUG Row 9:
Sucess
Same formula
Same Space group
Raw MP SG match: True
RAW LLM SG match: True

DEBUG Row 10:
Sucess
Same formula
Different space group

DEBUG Row 11:
Sucess
Same formula
Same Space group
Raw MP 

,material_input,llm_provider,llm_model,success,error_message,llm_formula,llm_space_group,llm_lattice_params,llm_atomic_positions,mp_formula,mp_space_group,mp_lattice_params,mp_atomic_positions,processing_time,api_failure,score
0,Silicon,google,gemini-2.5-flash,True,NaN,Si,Fd-3m,"{'a': 5.431, 'b': 5.431, 'c': 5.431, 'alpha': ...","[{'element': 'Si', 'position': [0.0, 0.0, 0.0]...",Si,Fd-3m,"{'a': 3.8492784033699095, 'b': 3.8492794116013...","[{'element': 'Si', 'position': [0.875, 0.875, ...",11.378151,False,1.0000
1,Diamond,google,gemini-2.5-flash,True,NaN,C,Fd-3m,"{'a': 3.567, 'b': 3.567, 'c': 3.567, 'alpha': ...","[{'element': 'C', 'position': [0.0, 0.0, 0.0]}...",C,P6_3/mmc,"{'a': 2.467291, 'b': 2.4672915232061654, 'c': ...","[{'element': 'C', 'position': [0.0, 0.0, 0.25]...",16.615215,False,0.5000
2,Graphite,google,gemini-2.5-flash,True,NaN,C,P6_3/mmc,"{'a': 2.461, 'b': 2.461, 'c': 6.708, 'alpha': ...","[{'element': 'C', 'position': [0.0, 0.0, 0.0]}...",C,P6_3/mmc,"{'a': 2.467291, 'b': 2.4672915232061654, 'c': ...","[{'element': 'C', 'position': [0.0, 0.0, 0.25]...",11.417326,False,0.9131
3,Aluminum,google,gemini-2.5-flash,True,NaN,Al,Fm-3m,"{'a': 4.05, 'b': 4.05, 'c': 4.05, 'alpha': 90,...","[{'element': 'Al', 'position': [0.0, 0.0, 0.0]}]",Al,Fm-3m,"{'a': 2.8559542459167657, 'b': 2.8559542916347...","[{'element': 'Al', 'position': [0.0, 0.0, 0.0]}]",14.766171,False,1.0000
4,Copper,google,gemini-2.5-flash,True,NaN,Cu,Fm-3m,"{'a': 3.615, 'b': 3.615, 'c': 3.615, 'alpha': ...","[{'element': 'Cu', 'position': [0.0, 0.0, 0.0]...",Cu,Fm-3m,"{'a': 2.5296252999953506, 'b': 2.5296258526342...","[{'element': 'Cu', 'position': [0.0, 0.0, 0.0]}]",8.915178,False,1.0000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
250,Potassium chloride,anthropic,claude-sonnet-4-6,True,NaN,KCl,Fm-3m,"{'a': 6.2931, 'b': 6.2931, 'c': 6.2931, 'alpha...","[{'element': 'K', 'position': [0.0, 0.0, 0.0]}...",KCl,Fm-3m,"{'a': 4.4432770640082975, 'b': 4.4432770147909...","[{'element': 'K', 'position': [0.0, 0.0, 0.0]}...",12.146904,False,1.0000
251,GST,anthropic,claude-sonnet-4-6,True,NaN,Ge2Sb2Te5,P-3m1,"{'a': 4.22, 'b': 4.22, 'c': 17.18, 'alpha': 90...","[{'element': 'Ge', 'position': [0.0, 0.0, 0.12...",Ge2Sb2Te5,P-3m1,"{'a': 4.29406442480606, 'b': 4.29406442480606,...","[{'element': 'Ge', 'position': [0.333333, 0.66...",14.793444,False,0.7500
252,Antimony trisulfide,anthropic,claude-sonnet-4-6,True,NaN,Sb2S3,Pnma,"{'a': 11.229, 'b': 3.836, 'c': 11.31, 'alpha':...","[{'element': 'Sb', 'position': [0.0662, 0.25, ...",Sb2S3,Pnma,"{'a': 3.8447329100000003, 'b': 11.36714114, 'c...","[{'element': 'Sb', 'position': [0.75, 0.465583...",19.754277,False,0.7500
253,Antimony triselenide,anthropic,claude-sonnet-4-6,True,NaN,Sb2Se3,Pnma,"{'a': 11.62, 'b': 3.985, 'c': 11.22, 'alpha': ...","[{'element': 'Sb', 'position': [0.0352, 0.25, ...",Sb2Se3,Pnma,"{'a': 4.00155687, 'b': 11.71362971, 'c': 12.36...","[{'element': 'Sb', 'position': [0.75, 0.038094...",18.596559,False,0.7500


# **Step 5- Final classification**


$$\text{Final Classification} = 0.9\times\text{Score} + 0.1\times\text{Average time} $$



In [71]:
# --- Base metrics ---
scores_series = df.groupby('llm_model')['score'].mean()
times_series  = df.groupby('llm_model')['processing_time'].mean()

# --- Final score ---
k = 1 / times_series.mean()
final_score = 0.80 * scores_series + 0.20 * np.exp(-k * times_series)

# --- Sort each independently ---
scores_series = scores_series.sort_values(ascending=False)
times_series  = times_series.sort_values(ascending=False)
final_score   = final_score.sort_values(ascending=False)

# --- Round ---
scores_series = scores_series.round(2)
times_series  = times_series.round(2)
final_score   = final_score.round(2)

# --- Convert to dicts ---
scores_dict = scores_series.to_dict()
times_dict  = times_series.to_dict()
final_dict  = final_score.to_dict()

print("Average Similarity Score")
print(scores_dict)
print()

print("Average Times")
print(times_dict)
print()

print("Final Classification")
print(final_dict)
print()

Average Similarity Score
{'claude-sonnet-4-6': 0.79, 'gemini-2.5-pro': 0.67, 'gemini-2.5-flash': 0.66, 'gpt-5.4': 0.61, 'gpt-5.4-mini': 0.4}

Average Times
{'gemini-2.5-pro': 37.05, 'gemini-2.5-flash': 18.4, 'claude-sonnet-4-6': 14.2, 'gpt-5.4': 5.81, 'gpt-5.4-mini': 2.99}

Final Classification
{'claude-sonnet-4-6': 0.71, 'gpt-5.4': 0.63, 'gemini-2.5-flash': 0.59, 'gemini-2.5-pro': 0.55, 'gpt-5.4-mini': 0.48}

